In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from src.preprocessing import preprocess_data
from src.features import add_structural_features
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/train.csv")

df = preprocess_data(df)
df = add_structural_features(df)

label_map = {
    "A": 0,
    "B": 1,
    "tie": 2
}

df["target"] = df["label"].map(label_map)

In [3]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=714,
    stratify=df["target"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=714,
    stratify=temp_df["target"]
)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(45981, 25)
(5748, 25)
(5748, 25)


# tokenize

In [10]:
import re

def tokenize(text):
    return re.findall(
        r"\b\w+\b",
        text.lower()
    )

In [11]:
def build_candidate_tokens(prompt_turns, response_turns):
    tokens = []

    for prompt, response in zip(prompt_turns, response_turns):

        tokens.append("<USER>")

        if isinstance(prompt, str):
            tokens.extend(tokenize(prompt))

        tokens.append("<ASSISTANT>")

        if isinstance(response, str):
            tokens.extend(tokenize(response))
        else:
            tokens.append("<EMPTY>")

    return tokens

In [12]:
train_df["candidate_a_tokens"] = train_df.apply(
    lambda row: build_candidate_tokens(
        row["prompt_list"],
        row["response_a_list"]
    ),
    axis=1
)

train_df["candidate_b_tokens"] = train_df.apply(
    lambda row: build_candidate_tokens(
        row["prompt_list"],
        row["response_b_list"]
    ),
    axis=1
)

In [13]:
sample = train_df[
    train_df["prompt_turns"] > 1
].iloc[0]

print(sample["candidate_a_tokens"][:100])

['<USER>', 'o', 'que', 'é', 'sincerismo', '<ASSISTANT>', 'sincerismo', 'é', 'um', 'termo', 'utilizado', 'em', 'filosofia', 'e', 'ética', 'para', 'designar', 'a', 'ideologia', 'que', 'defende', 'que', 'a', 'verdade', 'e', 'a', 'sinceridade', 'devem', 'ser', 'priorizadas', 'em', 'todas', 'as', 'interações', 'humanas', 'especialmente', 'em', 'relação', 'às', 'relações', 'interpessoais', 'o', 'sincerismo', 'argumenta', 'que', 'a', 'falta', 'de', 'sinceridade', 'e', 'a', 'mentira', 'são', 'danosas', 'às', 'relações', 'interpessoais', 'pois', 'erode', 'a', 'confiança', 'e', 'a', 'trustworthiness', 'entre', 'as', 'pessoas', 'por', 'isso', 'o', 'sincerismo', 'busca', 'promover', 'a', 'honestidade', 'a', 'transparência', 'e', 'a', 'sinceridade', 'em', 'todas', 'as', 'interações', 'humanas', 'desde', 'as', 'relações', 'pessoais', 'até', 'as', 'relações', 'políticas', 'e', 'sociais', 'o', 'sincerismo', 'também', 'pode', 'ser']


In [14]:
train_df["candidate_a_len"] = (
    train_df["candidate_a_tokens"].apply(len)
)

train_df["candidate_b_len"] = (
    train_df["candidate_b_tokens"].apply(len)
)

In [15]:
train_df[
    [
        "candidate_a_len",
        "candidate_b_len"
    ]
].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

,candidate_a_len,candidate_b_len
count,45981.000000,45981.000000
mean,276.409669,277.573519
std,301.374238,303.321811
min,4.000000,4.000000
50%,214.000000,215.000000
75%,350.000000,350.000000
90%,538.000000,542.000000
95%,740.000000,739.000000
99%,1500.000000,1508.200000
max,11106.000000,9209.000000


In [21]:
#head + tail truncation
MAX_LEN = 256

def truncate_tokens(tokens, max_len=MAX_LEN):
    if len(tokens) <= max_len:
        return tokens

    half = max_len // 2

    return tokens[:half] + tokens[-half:]

In [22]:
train_df["candidate_a_tokens_trunc"] = (
    train_df["candidate_a_tokens"].apply(truncate_tokens)
)

train_df["candidate_b_tokens_trunc"] = (
    train_df["candidate_b_tokens"].apply(truncate_tokens)
)

In [23]:
a_trunc_rate = (train_df["candidate_a_len"] > MAX_LEN).mean()
b_trunc_rate = (train_df["candidate_b_len"] > MAX_LEN).mean()

print(f"A truncation rate: {a_trunc_rate:.2%}")
print(f"B truncation rate: {b_trunc_rate:.2%}")

A truncation rate: 40.45%
B truncation rate: 40.86%


# Word2vec

In [28]:
from gensim.models import Word2Vec

In [29]:
w2v_sentences = (
    train_df["candidate_a_tokens_trunc"].tolist()
    + train_df["candidate_b_tokens_trunc"].tolist()
)

print(len(w2v_sentences))

91962


In [30]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=w2v_sentences,
    vector_size=64,
    window=5,
    min_count=2,
    workers=4,
    sg=0,
    epochs=3
)

In [31]:
print("Vocabulary size:", len(w2v_model.wv))
print("Embedding dimension:", w2v_model.vector_size)

Vocabulary size: 126566
Embedding dimension: 64


In [32]:
print("<USER>" in w2v_model.wv)
print("<ASSISTANT>" in w2v_model.wv)
print("<EMPTY>" in w2v_model.wv)

True
True
True


In [33]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

PAD_IDX = 0
UNK_IDX = 1

In [34]:
word_to_idx = {
    word: idx + 2
    for word, idx in w2v_model.wv.key_to_index.items()
}

word_to_idx[PAD_TOKEN] = PAD_IDX
word_to_idx[UNK_TOKEN] = UNK_IDX

In [35]:
print("Vocabulary size with PAD/UNK:", len(word_to_idx))

print("PAD:", word_to_idx["<PAD>"])
print("UNK:", word_to_idx["<UNK>"])
print("USER:", word_to_idx["<USER>"])
print("ASSISTANT:", word_to_idx["<ASSISTANT>"])

Vocabulary size with PAD/UNK: 126568
PAD: 0
UNK: 1
USER: 20
ASSISTANT: 22


In [36]:
def encode_tokens(tokens):
    return [
        word_to_idx.get(token, UNK_IDX)
        for token in tokens
    ]

In [37]:
sample_tokens = train_df["candidate_a_tokens_trunc"].iloc[0]

sample_ids = encode_tokens(sample_tokens)

print(sample_tokens[:20])
print(sample_ids[:20])
print(len(sample_tokens), len(sample_ids))

['<USER>', 'q2', 'the', 'body', 'ab', 'and', 'arm', 'bc', 'of', 'the', 'robotic', 'component', 'shown', 'in', 'figure', 'q2', 'rotates', 'at', 'a', 'constant']
[20, 11279, 2, 267, 4973, 3, 3122, 5501, 6, 2, 6853, 1067, 1903, 7, 1065, 11279, 12172, 46, 4, 1408]
256 256


In [38]:
#padding
def pad_sequence(ids, max_len=MAX_LEN):
    if len(ids) < max_len:
        ids = ids + [PAD_IDX] * (max_len - len(ids))

    return ids

In [39]:
def prepare_candidate_tokens(df):
    df = df.copy()

    df["candidate_a_tokens"] = df.apply(
        lambda row: build_candidate_tokens(
            row["prompt_list"],
            row["response_a_list"]
        ),
        axis=1
    )

    df["candidate_b_tokens"] = df.apply(
        lambda row: build_candidate_tokens(
            row["prompt_list"],
            row["response_b_list"]
        ),
        axis=1
    )

    df["candidate_a_tokens_trunc"] = (
        df["candidate_a_tokens"].apply(truncate_tokens)
    )

    df["candidate_b_tokens_trunc"] = (
        df["candidate_b_tokens"].apply(truncate_tokens)
    )

    return df

In [40]:
val_df = prepare_candidate_tokens(val_df)

In [41]:
def encode_and_pad(tokens):
    ids = encode_tokens(tokens)

    seq_len = min(len(ids), MAX_LEN)

    padded_ids = pad_sequence(ids)

    return padded_ids, seq_len

In [42]:
def prepare_encoded_data(df):
    a_ids = []
    b_ids = []
    a_lengths = []
    b_lengths = []

    for a_tokens, b_tokens in zip(
        df["candidate_a_tokens_trunc"],
        df["candidate_b_tokens_trunc"]
    ):
        a_padded, a_len = encode_and_pad(a_tokens)
        b_padded, b_len = encode_and_pad(b_tokens)

        a_ids.append(a_padded)
        b_ids.append(b_padded)

        a_lengths.append(a_len)
        b_lengths.append(b_len)

    return (
        np.array(a_ids, dtype=np.int32),
        np.array(b_ids, dtype=np.int32),
        np.array(a_lengths, dtype=np.int32),
        np.array(b_lengths, dtype=np.int32)
    )

In [43]:
X_train_a, X_train_b, len_train_a, len_train_b = (
    prepare_encoded_data(train_df)
)

X_val_a, X_val_b, len_val_a, len_val_b = (
    prepare_encoded_data(val_df)
)

In [44]:
print(X_train_a.shape)
print(X_train_b.shape)
print(len_train_a.shape)

print(X_val_a.shape)
print(X_val_b.shape)

(45981, 256)
(45981, 256)
(45981,)
(5748, 256)
(5748, 256)


In [46]:
#matrix
EMBED_DIM = 64

embedding_matrix = np.zeros(
    (len(word_to_idx), EMBED_DIM),
    dtype=np.float32
)

for word in w2v_model.wv.key_to_index:
    idx = word_to_idx[word]
    embedding_matrix[idx] = w2v_model.wv[word]
    
embedding_matrix[UNK_IDX] = (
    w2v_model.wv.vectors.mean(axis=0)
)

print(embedding_matrix.shape)

(126568, 64)


In [47]:
import torch
from torch.utils.data import Dataset, DataLoader

In [48]:
class PreferenceDataset(Dataset):
    def __init__(
        self,
        a_ids,
        b_ids,
        a_lengths,
        b_lengths,
        labels
    ):
        self.a_ids = a_ids
        self.b_ids = b_ids
        self.a_lengths = a_lengths
        self.b_lengths = b_lengths
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "a_ids": torch.tensor(
                self.a_ids[idx],
                dtype=torch.long
            ),
            "b_ids": torch.tensor(
                self.b_ids[idx],
                dtype=torch.long
            ),
            "a_length": torch.tensor(
                self.a_lengths[idx],
                dtype=torch.long
            ),
            "b_length": torch.tensor(
                self.b_lengths[idx],
                dtype=torch.long
            ),
            "label": torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }

In [49]:
train_dataset = PreferenceDataset(
    X_train_a,
    X_train_b,
    len_train_a,
    len_train_b,
    train_df["target"].values
)

val_dataset = PreferenceDataset(
    X_val_a,
    X_val_b,
    len_val_a,
    len_val_b,
    val_df["target"].values
)

In [66]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# LSTM

In [51]:
EMBED_DIM = 64
HIDDEN_DIM = 64

device = torch.device("cpu")

In [56]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence


class PreferenceLSTM(nn.Module):
    def __init__(
        self,
        embedding_matrix,
        hidden_dim=64
    ):
        super().__init__()

        vocab_size, embed_dim = embedding_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(
                embedding_matrix,
                dtype=torch.float32
            ),
            freeze=True,
            padding_idx=PAD_IDX
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 4, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 3)
        )
        
    def encode(self, ids, lengths):
        embedded = self.embedding(ids)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.lstm(packed)

        return hidden[-1]

    def forward(
        self,
        a_ids,
        b_ids,
        a_lengths,
        b_lengths
    ):
        h_a = self.encode(a_ids, a_lengths)
        h_b = self.encode(b_ids, b_lengths)

        combined = torch.cat(
            [
                h_a,
                h_b,
                h_a - h_b,
                torch.abs(h_a - h_b)
            ],
            dim=1
        )

        logits = self.classifier(combined)

        return logits

In [67]:
model = PreferenceLSTM(
    embedding_matrix=embedding_matrix,
    hidden_dim=HIDDEN_DIM
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [68]:
def train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    device
):
    model.train()

    total_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        a_ids = batch["a_ids"].to(device)
        b_ids = batch["b_ids"].to(device)
        a_lengths = batch["a_length"].to(device)
        b_lengths = batch["b_length"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(
            a_ids,
            b_ids,
            a_lengths,
            b_lengths
        )

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (batch_idx + 1) % 100 == 0:
            print(
                f"Batch {batch_idx + 1}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    return total_loss / len(train_loader)

In [69]:
from sklearn.metrics import log_loss

def evaluate_model(
    model,
    data_loader,
    device
):
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            a_ids = batch["a_ids"].to(device)
            b_ids = batch["b_ids"].to(device)
            a_lengths = batch["a_length"].to(device)
            b_lengths = batch["b_length"].to(device)

            logits = model(
                a_ids,
                b_ids,
                a_lengths,
                b_lengths
            )

            probs = torch.softmax(
                logits,
                dim=1
            )

            all_probs.append(
                probs.cpu().numpy()
            )

            all_labels.append(
                batch["label"].numpy()
            )

    probs = np.vstack(all_probs)
    labels = np.concatenate(all_labels)

    loss = log_loss(
        labels,
        probs,
        labels=[0, 1, 2]
    )

    return loss, probs

In [70]:
EPOCHS = 3

best_val_loss = float("inf")
best_model_state = None

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, _ = evaluate_model(
        model,
        val_loader,
        device
    )

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Log Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        best_model_state = {
            key: value.cpu().clone()
            for key, value in model.state_dict().items()
        }


Epoch 1/3
Batch 100/719 | Loss: 1.0826
Batch 200/719 | Loss: 1.0970
Batch 300/719 | Loss: 1.0949
Batch 400/719 | Loss: 1.0816
Batch 500/719 | Loss: 1.0700
Batch 600/719 | Loss: 1.0718
Batch 700/719 | Loss: 1.1288
Train Loss: 1.0842 | Validation Log Loss: 1.0812

Epoch 2/3
Batch 100/719 | Loss: 1.0231
Batch 200/719 | Loss: 1.0629
Batch 300/719 | Loss: 1.0376
Batch 400/719 | Loss: 1.0578
Batch 500/719 | Loss: 1.0967
Batch 600/719 | Loss: 1.1106
Batch 700/719 | Loss: 1.0200
Train Loss: 1.0611 | Validation Log Loss: 1.0587

Epoch 3/3
Batch 100/719 | Loss: 1.0516
Batch 200/719 | Loss: 1.0121
Batch 300/719 | Loss: 1.1090
Batch 400/719 | Loss: 1.0332
Batch 500/719 | Loss: 1.0319
Batch 600/719 | Loss: 1.0069
Batch 700/719 | Loss: 1.0242
Train Loss: 1.0419 | Validation Log Loss: 1.0559


In [71]:
model.load_state_dict(best_model_state)

lstm_val_logloss, lstm_val_probs = evaluate_model(
    model,
    val_loader,
    device
)

print(
    f"Best LSTM Validation Log Loss: "
    f"{lstm_val_logloss:.4f}"
)

Best LSTM Validation Log Loss: 1.0559


In [62]:
model.train()

total_loss = 0

for batch_idx, batch in enumerate(smoke_loader):
    a_ids = batch["a_ids"].to(device)
    b_ids = batch["b_ids"].to(device)
    a_lengths = batch["a_length"].to(device)
    b_lengths = batch["b_length"].to(device)
    labels = batch["label"].to(device)

    optimizer.zero_grad()

    logits = model(
        a_ids,
        b_ids,
        a_lengths,
        b_lengths
    )

    loss = criterion(
        logits,
        labels
    )

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

    if (batch_idx + 1) % 10 == 0:
        print(
            f"Batch {batch_idx + 1}/{len(smoke_loader)}, "
            f"Loss: {loss.item():.4f}"
        )

average_train_loss = total_loss / len(smoke_loader)

print(
    f"Average training loss: "
    f"{average_train_loss:.4f}"
)

Batch 10/79, Loss: 1.0898
Batch 20/79, Loss: 1.0959
Batch 30/79, Loss: 1.0959
Batch 40/79, Loss: 1.0992
Batch 50/79, Loss: 1.0952
Batch 60/79, Loss: 1.0971
Batch 70/79, Loss: 1.1035
Average training loss: 1.0964


In [63]:
all_probs = []
all_labels = []

model.eval()

with torch.no_grad():

    for batch in val_loader:
        a_ids = batch["a_ids"].to(device)
        b_ids = batch["b_ids"].to(device)
        a_lengths = batch["a_length"].to(device)
        b_lengths = batch["b_length"].to(device)

        logits = model(
            a_ids,
            b_ids,
            a_lengths,
            b_lengths
        )

        probs = torch.softmax(
            logits,
            dim=1
        )

        all_probs.append(
            probs.cpu().numpy()
        )

        all_labels.append(
            batch["label"].numpy()
        )

In [64]:
val_probs = np.vstack(all_probs)
val_labels = np.concatenate(all_labels)

print(val_probs.shape)
print(val_labels.shape)

(5748, 3)
(5748,)


In [65]:
from sklearn.metrics import log_loss

lstm_smoke_logloss = log_loss(
    val_labels,
    val_probs,
    labels=[0, 1, 2]
)

print(
    f"LSTM smoke-test Validation Log Loss: "
    f"{lstm_smoke_logloss:.4f}"
)

LSTM smoke-test Validation Log Loss: 1.0943
